# 401 — Consensus Program Construction

## Objective

Construct symmetric candidate cross-system transcriptomic consensus representations from the `SUPPORTED_CORRESPONDENCE` events frozen by notebook 400.

Each consensus representation combines the independently discovered tumor and cell-line transcriptomic loadings over the frozen shared-gene universe while preserving tumor-side methylation arms, structural-family dependence, and system-specific robustness and limitation metadata as contextual evidence.

Consensus structure is defined without using pharmacogenomic phenotype, downstream biological annotation, internal robustness strength, scientific priority, or tumor-side methylation information to select, orient, or weight genes.

The frozen Phase 4 cell-line candidate space was inherited from Phase 3, where phenotype association contributed to candidate definition. Notebook 401 does not reuse pharmacogenomic phenotype information to orient, weight, score, rescue, or reoptimize supported cross-system correspondences.
 
The resulting objects are candidate cross-system transcriptomic consensus representations with tumor-side methylation context. They do not establish full epigenetic-transcriptomic reproduction, biological causality, independent validation, clinical prediction, or therapeutic relevance.

In [1]:
# =============================================================================
# Imports
# =============================================================================

import json

import h5py
import numpy as np
import pandas as pd

from scipy.stats import pearsonr, spearmanr

from pancancer_epigenetics.utils.paths import (
    Paths,
    project_relative_path,
)

In [2]:
# =============================================================================
# Input and output directories
# =============================================================================

TUMOR_PROGRAM_DIR = Paths.tumor_programs
CELL_LINE_PROGRAM_DIR = Paths.cellline_programs
OUTPUT_DIR = Paths.consensus_programs

In [3]:
# =============================================================================
# Authoritative consensus-construction input paths
# =============================================================================

TUMOR_HANDOFF_PATH = (
    Paths.config
    / "program_handoffs"
    / "tcga_phase2_candidate_program_handoff.csv"
)

CORRESPONDENCE_SUMMARY_PATH = (
    OUTPUT_DIR
    / "400_cross_system_correspondence_summary.csv"
)

SHARED_GENE_UNIVERSE_PATH = (
    OUTPUT_DIR
    / "400_cross_system_shared_gene_universe.csv"
)

TUMOR_RNA_LOADINGS_PATH = (
    TUMOR_PROGRAM_DIR
    / "tcga_primary_tumor_rna_ica_candidate_gene_loadings.csv"
)

CELL_LINE_ICA_LOADINGS_PATH = (
    CELL_LINE_PROGRAM_DIR
    / "310_ica_program_loadings.parquet"
)

CROSS_SYSTEM_TUMOR_ARM_HANDOFF_PATH = (
    OUTPUT_DIR
    / "400_cross_system_tumor_arm_handoff.csv"
)

In [4]:
# =============================================================================
# Load authoritative consensus-construction inputs
# =============================================================================

tumor_handoff = pd.read_csv(TUMOR_HANDOFF_PATH)

correspondence_summary = pd.read_csv(
    CORRESPONDENCE_SUMMARY_PATH
)

shared_gene_universe = pd.read_csv(
    SHARED_GENE_UNIVERSE_PATH
)

tumor_rna_loadings = pd.read_csv(
    TUMOR_RNA_LOADINGS_PATH
)

cell_line_ica_loadings = pd.read_parquet(
    CELL_LINE_ICA_LOADINGS_PATH
)

cross_system_tumor_arm_handoff = pd.read_csv(
    CROSS_SYSTEM_TUMOR_ARM_HANDOFF_PATH
)



In [5]:
# =============================================================================
# Define construction-eligible transcriptomic correspondence events
# =============================================================================

construction_events = (
    correspondence_summary
    .loc[
        correspondence_summary["correspondence_class"]
        .eq("SUPPORTED_CORRESPONDENCE"),
        [
            "tumor_rna_axis",
            "cell_line_program",
            "orientation_multiplier",
            "tumor_arm_count",
            "tumor_candidate_pairs",
            "robustness_category",
            "program_status",
            "context_sensitive",
            "association_unstable",
            "unresolved_confounding",
            "cross_method_convergent",
        ],
    ]
    .sort_values("tumor_rna_axis")
    .reset_index(drop=True)
)

construction_events

,tumor_rna_axis,cell_line_program,orientation_multiplier,tumor_arm_count,tumor_candidate_pairs,robustness_category,program_status,context_sensitive,association_unstable,unresolved_confounding,cross_method_convergent
0,RNA_IC150,ICA_PROGRAM_09,1,1,CROSS_OMIC_PAIR_04,CONTEXT_SENSITIVE_CANDIDATE,candidate_with_cross_method_support,True,False,False,True
1,RNA_IC151,ICA_PROGRAM_29,-1,1,CROSS_OMIC_PAIR_08,ROBUSTNESS_SUPPORTED_CANDIDATE,candidate_with_cross_method_support,False,False,False,True
2,RNA_IC184,ICA_PROGRAM_13,-1,2,CROSS_OMIC_PAIR_03;CROSS_OMIC_PAIR_12,ROBUSTNESS_SUPPORTED_CANDIDATE,candidate_ica_specific,False,False,False,False


In [6]:
# =============================================================================
# Assign frozen consensus transcriptomic program IDs
# =============================================================================

# IDs follow deterministic tumor RNA-axis order rather than any diagnostic,
# robustness, biological, or scientific-priority ranking.

construction_events = construction_events.assign(
    consensus_program_id=[
        f"CONSENSUS_TX_{index:02d}"
        for index in range(1, len(construction_events) + 1)
    ]
)

construction_events[
    [
        "consensus_program_id",
        "tumor_rna_axis",
        "cell_line_program",
        "orientation_multiplier",
        "tumor_candidate_pairs",
    ]
]

,consensus_program_id,tumor_rna_axis,cell_line_program,orientation_multiplier,tumor_candidate_pairs
0,CONSENSUS_TX_01,RNA_IC150,ICA_PROGRAM_09,1,CROSS_OMIC_PAIR_04
1,CONSENSUS_TX_02,RNA_IC151,ICA_PROGRAM_29,-1,CROSS_OMIC_PAIR_08
2,CONSENSUS_TX_03,RNA_IC184,ICA_PROGRAM_13,-1,CROSS_OMIC_PAIR_03;CROSS_OMIC_PAIR_12


In [7]:
# =============================================================================
# Align source loadings to the frozen shared-gene universe
# =============================================================================

# Reuse the exact cross-system identifiers and gene order frozen in notebook
# 400; no re-harmonization, zero-filling, or feature reselection is introduced.

tumor_shared_loadings = (
    shared_gene_universe[
        ["gene_symbol", "tumor_gene_id"]
    ]
    .merge(
        tumor_rna_loadings,
        left_on="tumor_gene_id",
        right_on="gene_id",
        how="left",
        sort=False,
    )
    [
        ["gene_symbol", *construction_events["tumor_rna_axis"]]
    ]
)

cell_line_shared_loadings = (
    shared_gene_universe[
        ["gene_symbol", "cell_line_gene_id"]
    ]
    .merge(
        cell_line_ica_loadings,
        left_on="cell_line_gene_id",
        right_on="gene",
        how="left",
        sort=False,
    )
    [
        ["gene_symbol", *construction_events["cell_line_program"]]
    ]
)

In [8]:
# =============================================================================
# Verify frozen shared-gene alignment
# =============================================================================

{
    "shared_gene_count": len(shared_gene_universe),
    "tumor_aligned_gene_count": len(tumor_shared_loadings),
    "cell_line_aligned_gene_count": len(cell_line_shared_loadings),
    "tumor_gene_order_preserved": tumor_shared_loadings["gene_symbol"].equals(
        shared_gene_universe["gene_symbol"]
    ),
    "cell_line_gene_order_preserved": cell_line_shared_loadings["gene_symbol"].equals(
        shared_gene_universe["gene_symbol"]
    ),
    "tumor_missing_loadings": int(
        tumor_shared_loadings[
            construction_events["tumor_rna_axis"]
        ].isna().sum().sum()
    ),
    "cell_line_missing_loadings": int(
        cell_line_shared_loadings[
            construction_events["cell_line_program"]
        ].isna().sum().sum()
    ),
}

{'shared_gene_count': 2389,
 'tumor_aligned_gene_count': 2389,
 'cell_line_aligned_gene_count': 2389,
 'tumor_gene_order_preserved': True,
 'cell_line_gene_order_preserved': True,
 'tumor_missing_loadings': 0,
 'cell_line_missing_loadings': 0}

In [9]:
# =============================================================================
# Define symmetric consensus-loading construction
# =============================================================================

def build_consensus_loading(
    tumor_loading,
    cell_line_loading,
    orientation_multiplier,
):
    """Construct one symmetric tumor–cell-line consensus loading."""

    tumor_centered = tumor_loading - tumor_loading.mean()

    # The multiplier is frozen in notebook 400 to resolve ICA sign
    # indeterminacy; it is not phenotype- or biology-oriented.
    cell_line_oriented = orientation_multiplier * cell_line_loading
    cell_line_centered = cell_line_oriented - cell_line_oriented.mean()

    tumor_normalized = tumor_centered / np.linalg.norm(tumor_centered)
    cell_line_normalized = cell_line_centered / np.linalg.norm(cell_line_centered)

    # Equal system weighting is prespecified; robustness, priority, phenotype,
    # and tumor methylation do not modify consensus structure.
    consensus_loading = (
        tumor_normalized + cell_line_normalized
    ) / 2

    return consensus_loading / np.linalg.norm(consensus_loading)

In [10]:
# =============================================================================
# Construct consensus transcriptomic gene weights
# =============================================================================

consensus_weight_frames = []

for event in construction_events.itertuples(index=False):
    tumor_loading = tumor_shared_loadings[
        event.tumor_rna_axis
    ].to_numpy(dtype=float)

    cell_line_loading = cell_line_shared_loadings[
        event.cell_line_program
    ].to_numpy(dtype=float)

    consensus_loading = build_consensus_loading(
        tumor_loading=tumor_loading,
        cell_line_loading=cell_line_loading,
        orientation_multiplier=event.orientation_multiplier,
    )

    consensus_weight_frames.append(
        pd.DataFrame(
            {
                "consensus_program_id": event.consensus_program_id,
                "gene_symbol": shared_gene_universe["gene_symbol"],
                "tumor_rna_axis": event.tumor_rna_axis,
                "cell_line_program": event.cell_line_program,
                "orientation_multiplier": event.orientation_multiplier,
                "tumor_loading": tumor_loading,
                "cell_line_loading": cell_line_loading,
                "consensus_weight": consensus_loading,
            }
        )
    )

consensus_gene_weights = pd.concat(
    consensus_weight_frames,
    ignore_index=True,
)

In [11]:
# =============================================================================
# Verify consensus gene-weight construction
# =============================================================================

consensus_weight_norms = (
    consensus_gene_weights
    .groupby("consensus_program_id")["consensus_weight"]
    .apply(lambda weights: np.linalg.norm(weights.to_numpy()))
)

{
    "consensus_programs": consensus_gene_weights[
        "consensus_program_id"
    ].nunique(),
    "rows": len(consensus_gene_weights),
    "expected_rows": (
        len(shared_gene_universe)
        * len(construction_events)
    ),
    "genes_per_program": consensus_gene_weights.groupby(
        "consensus_program_id"
    )["gene_symbol"].size().to_dict(),
    "missing_consensus_weights": int(
        consensus_gene_weights["consensus_weight"].isna().sum()
    ),
    "consensus_l2_norms": consensus_weight_norms.to_dict(),
}

{'consensus_programs': 3,
 'rows': 7167,
 'expected_rows': 7167,
 'genes_per_program': {'CONSENSUS_TX_01': 2389,
  'CONSENSUS_TX_02': 2389,
  'CONSENSUS_TX_03': 2389},
 'missing_consensus_weights': 0,
 'consensus_l2_norms': {'CONSENSUS_TX_01': 1.0,
  'CONSENSUS_TX_02': 0.9999999999999999,
  'CONSENSUS_TX_03': 1.0000000000000002}}

In [12]:
# =============================================================================
# Calculate shared-loading energy diagnostics
# =============================================================================

# Shared-loading energy is descriptive only: it quantifies representation of
# each original 5,000-gene loading in the frozen shared universe and is not a gate.

shared_loading_energy = pd.DataFrame(
    [
        {
            "consensus_program_id": event.consensus_program_id,
            "tumor_rna_axis": event.tumor_rna_axis,
            "cell_line_program": event.cell_line_program,
            "tumor_shared_loading_energy": (
                np.square(
                    tumor_shared_loadings[event.tumor_rna_axis].to_numpy()
                ).sum()
                / np.square(
                    tumor_rna_loadings[event.tumor_rna_axis].to_numpy()
                ).sum()
            ),
            "cell_line_shared_loading_energy": (
                np.square(
                    cell_line_shared_loadings[event.cell_line_program].to_numpy()
                ).sum()
                / np.square(
                    cell_line_ica_loadings[event.cell_line_program].to_numpy()
                ).sum()
            ),
        }
        for event in construction_events.itertuples(index=False)
    ]
)

shared_loading_energy

,consensus_program_id,tumor_rna_axis,cell_line_program,tumor_shared_loading_energy,cell_line_shared_loading_energy
0,CONSENSUS_TX_01,RNA_IC150,ICA_PROGRAM_09,0.543179,0.410864
1,CONSENSUS_TX_02,RNA_IC151,ICA_PROGRAM_29,0.550136,0.637620
2,CONSENSUS_TX_03,RNA_IC184,ICA_PROGRAM_13,0.532977,0.544084


In [13]:
# =============================================================================
# Build tumor-arm context for supported consensus events
# =============================================================================

# Tumor methylation arms remain separate contextual units. Arms sharing one RNA
# axis are not fused and do not count as independent transcriptomic events.
tumor_robustness_context_columns = [
    "lopo_absolute_shift",
    "lopo_direction_preserved_all",
    "bootstrap_direction_preservation_fraction",
    "confounder_maximum_absolute_shift",
    "confounder_direction_preserved_all",
    "plate_median_absolute_shift",
    "plate_project_direction_preservation_fraction",
    "rna_seed_median",
    "rna_subsample_median",
    "methylation_seed_median",
    "methylation_subsample_median",
    "minimum_nmf_concordance",
]

tumor_arm_context = (
    tumor_handoff
    .merge(
        construction_events[
            [
                "consensus_program_id",
                "tumor_rna_axis",
                "cell_line_program",
                "orientation_multiplier",
            ]
        ],
        left_on="rna_component",
        right_on="tumor_rna_axis",
        how="inner",
        validate="many_to_one",
    )
    .merge(
        cross_system_tumor_arm_handoff[
            [
                "candidate_pair",
                *tumor_robustness_context_columns,
            ]
        ],
        on="candidate_pair",
        how="left",
        validate="one_to_one",
    )
    [
        [
            "consensus_program_id",
            "candidate_pair",
            "rna_component",
            "methylation_component",
            "structural_family",
            "cell_line_program",
            "orientation_multiplier",
            "final_audit_status",
            "scientific_priority",
            "strict_hm27_status",
            "relationship_form",
            "primary_limitation",
            "consensus_handling",
            *tumor_robustness_context_columns,
        ]
    ]
    .sort_values(
        ["consensus_program_id", "candidate_pair"]
    )
    .reset_index(drop=True)
)

tumor_arm_context

,consensus_program_id,candidate_pair,rna_component,methylation_component,structural_family,cell_line_program,orientation_multiplier,final_audit_status,scientific_priority,strict_hm27_status,...,bootstrap_direction_preservation_fraction,confounder_maximum_absolute_shift,confounder_direction_preserved_all,plate_median_absolute_shift,plate_project_direction_preservation_fraction,rna_seed_median,rna_subsample_median,methylation_seed_median,methylation_subsample_median,minimum_nmf_concordance
0,CONSENSUS_TX_01,CROSS_OMIC_PAIR_04,RNA_IC150,METH_IC128,PF01,ICA_PROGRAM_09,1,CONFOUNDED,LOW,SUPPORTED_SMALL_EFFECT,...,1.0,0.056843,True,0.002370,0.962963,0.989194,0.980672,0.972962,0.948443,0.316420
1,CONSENSUS_TX_02,CROSS_OMIC_PAIR_08,RNA_IC151,METH_IC050,PF08,ICA_PROGRAM_29,-1,METHOD_LIMITED,MEDIUM,NOT_EVALUABLE,...,1.0,0.008919,True,0.006776,1.000000,0.623267,0.583573,0.312480,0.337208,0.089150
2,CONSENSUS_TX_03,CROSS_OMIC_PAIR_03,RNA_IC184,METH_IC169,PF01,ICA_PROGRAM_13,-1,METHOD_LIMITED,CONDITIONAL_HIGH,NOT_EVALUABLE,...,1.0,0.006924,True,0.006108,1.000000,0.940041,0.950575,0.462809,0.538696,0.103034
3,CONSENSUS_TX_03,CROSS_OMIC_PAIR_12,RNA_IC184,METH_IC128,PF01,ICA_PROGRAM_13,-1,CONFOUNDED,LOW_CONDITIONAL,NOT_EVALUABLE,...,1.0,0.021054,True,0.018035,1.000000,0.940041,0.950575,0.972962,0.948443,0.119982


In [14]:
# =============================================================================
# Build consensus transcriptomic program catalog
# =============================================================================

tumor_context_summary = (
    tumor_arm_context
    .groupby("consensus_program_id", as_index=False)
    .agg(
        tumor_candidate_pairs=(
            "candidate_pair",
            lambda values: ";".join(values),
        ),
        tumor_methylation_components=(
            "methylation_component",
            lambda values: ";".join(values),
        ),
        tumor_structural_families=(
            "structural_family",
            lambda values: ";".join(dict.fromkeys(values)),
        ),
    )
)

consensus_program_catalog = (
    construction_events[
        [
            "consensus_program_id",
            "tumor_rna_axis",
            "cell_line_program",
            "orientation_multiplier",
            "tumor_arm_count",
            "robustness_category",
            "program_status",
            "context_sensitive",
            "association_unstable",
            "unresolved_confounding",
            "cross_method_convergent",
        ]
    ]
    .merge(
        tumor_context_summary,
        on="consensus_program_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        shared_loading_energy,
        on=[
            "consensus_program_id",
            "tumor_rna_axis",
            "cell_line_program",
        ],
        how="left",
        validate="one_to_one",
    )
    .assign(
        representation_scope=(
            "candidate_cross_system_transcriptomic_representation_"
            "with_tumor_side_methylation_context"
        )
    )
)

consensus_program_catalog

,consensus_program_id,tumor_rna_axis,cell_line_program,orientation_multiplier,tumor_arm_count,robustness_category,program_status,context_sensitive,association_unstable,unresolved_confounding,cross_method_convergent,tumor_candidate_pairs,tumor_methylation_components,tumor_structural_families,tumor_shared_loading_energy,cell_line_shared_loading_energy,representation_scope
0,CONSENSUS_TX_01,RNA_IC150,ICA_PROGRAM_09,1,1,CONTEXT_SENSITIVE_CANDIDATE,candidate_with_cross_method_support,True,False,False,True,CROSS_OMIC_PAIR_04,METH_IC128,PF01,0.543179,0.410864,candidate_cross_system_transcriptomic_represen...
1,CONSENSUS_TX_02,RNA_IC151,ICA_PROGRAM_29,-1,1,ROBUSTNESS_SUPPORTED_CANDIDATE,candidate_with_cross_method_support,False,False,False,True,CROSS_OMIC_PAIR_08,METH_IC050,PF08,0.550136,0.637620,candidate_cross_system_transcriptomic_represen...
2,CONSENSUS_TX_03,RNA_IC184,ICA_PROGRAM_13,-1,2,ROBUSTNESS_SUPPORTED_CANDIDATE,candidate_ica_specific,False,False,False,False,CROSS_OMIC_PAIR_03;CROSS_OMIC_PAIR_12,METH_IC169;METH_IC128,PF01,0.532977,0.544084,candidate_cross_system_transcriptomic_represen...


In [15]:
# =============================================================================
# Score-projection and native-score input paths
# =============================================================================

TUMOR_EXPRESSION_PATH = (
    Paths.expression
    / "tcga_primary_tumor_rnaseq_program_discovery_tmm_logcpm.h5"
)

TUMOR_NATIVE_SCORES_PATH = (
    TUMOR_PROGRAM_DIR
    / "tcga_primary_tumor_rna_ica_candidate_scores.csv"
)

# Use the expression-only Phase 3 artifact rather than the integrated
# transcriptome–phenotype table so phenotype is not carried into score projection.

CELL_LINE_EXPRESSION_PATH = (
    Paths.expression
    / "303_expression_harmonized.parquet"
)

CELL_LINE_NATIVE_SCORES_PATH = (
    CELL_LINE_PROGRAM_DIR
    / "310_ica_program_scores.parquet"
)

In [16]:
# =============================================================================
# Load score-projection and native-score inputs
# =============================================================================

tumor_expression_h5 = h5py.File(
    TUMOR_EXPRESSION_PATH,
    "r",
)
tumor_expression = tumor_expression_h5["logcpm"]

tumor_native_scores = pd.read_csv(
    TUMOR_NATIVE_SCORES_PATH
)

cell_line_expression = pd.read_parquet(
    CELL_LINE_EXPRESSION_PATH
)

cell_line_native_scores = pd.read_parquet(
    CELL_LINE_NATIVE_SCORES_PATH
)

In [17]:
# =============================================================================
# Prepare frozen shared-gene selectors for score projection
# =============================================================================

# Projection reuses the exact notebook-400 crosswalk and gene order; identifiers
# are not re-harmonized or reselected at this stage.

tumor_projection_index = (
    shared_gene_universe[
        ["gene_symbol", "tumor_gene_id"]
    ]
    .merge(
        tumor_rna_loadings[
            ["gene_id", "filtered_matrix_row_index"]
        ],
        left_on="tumor_gene_id",
        right_on="gene_id",
        how="left",
        sort=False,
    )
)

tumor_projection_rows = tumor_projection_index[
    "filtered_matrix_row_index"
].to_numpy(dtype=int)

cell_line_projection_columns = shared_gene_universe[
    "cell_line_gene_id"
].tolist()

In [18]:
# =============================================================================
# Verify shared-gene projection selectors
# =============================================================================

{
    "tumor_projection_gene_count": len(tumor_projection_rows),
    "tumor_rows_strictly_increasing": bool(
        np.all(np.diff(tumor_projection_rows) > 0)
    ),
    "cell_line_projection_gene_count": len(cell_line_projection_columns),
    "cell_line_columns_available": all(
        column in cell_line_expression.columns
        for column in cell_line_projection_columns
    ),
}

{'tumor_projection_gene_count': 2389,
 'tumor_rows_strictly_increasing': False,
 'cell_line_projection_gene_count': 2389,
 'cell_line_columns_available': True}

In [19]:
# =============================================================================
# Prepare HDF5-compatible tumor row order
# =============================================================================

# h5py requires increasing advanced-index rows. Sort only for physical reading,
# then restore the frozen notebook-400 shared-gene order immediately afterward.

tumor_h5_read_order = np.argsort(
    tumor_projection_rows
)

tumor_h5_rows = tumor_projection_rows[
    tumor_h5_read_order
]

tumor_shared_restore_order = np.argsort(
    tumor_h5_read_order
)

In [20]:
# =============================================================================
# Define within-system consensus-score projection
# =============================================================================

def project_consensus_scores(expression_matrix, consensus_weights):
    """Project consensus loadings and standardize each score within one system."""

    # Expression is standardized independently within each biological system;
    # TCGA and cell-line distributions are never jointly normalized.
    gene_means = expression_matrix.mean(axis=0, keepdims=True)
    gene_stds = expression_matrix.std(axis=0, ddof=1, keepdims=True)

    standardized_expression = (
        expression_matrix - gene_means
    ) / gene_stds

    projected_scores = standardized_expression @ consensus_weights

    # Each consensus score is scaled independently within the same system.
    score_means = projected_scores.mean(axis=0, keepdims=True)
    score_stds = projected_scores.std(axis=0, ddof=1, keepdims=True)

    return (
        projected_scores - score_means
    ) / score_stds

In [21]:
# =============================================================================
# Prepare consensus weight matrix for score projection
# =============================================================================

# Restore the frozen gene order and deterministic consensus-program order before
# matrix projection.

consensus_weight_matrix = (
    consensus_gene_weights
    .pivot(
        index="gene_symbol",
        columns="consensus_program_id",
        values="consensus_weight",
    )
    .reindex(shared_gene_universe["gene_symbol"])
)

consensus_weight_matrix = consensus_weight_matrix[
    construction_events["consensus_program_id"]
]

In [22]:
# =============================================================================
# Extract shared-gene expression matrices
# =============================================================================

# TCGA rows are read in HDF5-compatible order and immediately restored to the
# frozen shared-gene order defined in notebook 400.
tumor_shared_expression = (
    tumor_expression[
        tumor_h5_rows,
        :,
    ][
        tumor_shared_restore_order,
        :
    ]
)

# Cell-line columns already follow the frozen cross-system gene identifiers.
cell_line_shared_expression = (
    cell_line_expression[
        cell_line_projection_columns
    ]
    .to_numpy(dtype=np.float64)
)

# The selected TCGA matrix is now materialized in memory; the HDF5 handle is no
# longer required downstream.
tumor_expression_h5.close()

In [23]:
# =============================================================================
# Project tumor consensus transcriptomic scores
# =============================================================================

# The frozen TCGA expression representation is genes × samples; transpose only
# for projection so that genes remain aligned with the consensus weight matrix.
tumor_consensus_score_values = project_consensus_scores(
    tumor_shared_expression.T,
    consensus_weight_matrix.to_numpy(dtype=np.float64),
)

In [24]:
# =============================================================================
# Project cell-line consensus transcriptomic scores
# =============================================================================

# The frozen cell-line expression matrix is already samples × genes, matching
# the orientation required for projection against the consensus weight matrix.
cell_line_consensus_score_values = project_consensus_scores(
    cell_line_shared_expression,
    consensus_weight_matrix.to_numpy(dtype=np.float64),
)

In [25]:
# =============================================================================
# Build tumor consensus-score table
# =============================================================================

# Native Phase 2 scores provide the frozen sample metadata and sample order only;
# their ICA score values do not enter consensus-score construction.
tumor_score_metadata_columns = [
    column
    for column in tumor_native_scores.columns
    if not column.startswith("RNA_IC")
]

tumor_consensus_scores = pd.concat(
    [
        tumor_native_scores[
            tumor_score_metadata_columns
        ].reset_index(drop=True),
        pd.DataFrame(
            tumor_consensus_score_values,
            columns=consensus_weight_matrix.columns,
        ),
    ],
    axis=1,
)

In [26]:
# =============================================================================
# Build cell-line consensus-score table
# =============================================================================

# Model identity and row order come directly from the frozen transcriptomic
# representation; no pharmacological phenotype enters the consensus artifact.
cell_line_consensus_scores = pd.concat(
    [
        cell_line_expression[
            ["ModelID"]
        ].reset_index(drop=True),
        pd.DataFrame(
            cell_line_consensus_score_values,
            columns=consensus_weight_matrix.columns,
        ),
    ],
    axis=1,
)

In [27]:
# =============================================================================
# Verify consensus-score construction
# =============================================================================

tumor_score_columns = consensus_weight_matrix.columns.tolist()
cell_line_score_columns = consensus_weight_matrix.columns.tolist()

{
    "tumor_shape": tumor_consensus_scores.shape,
    "cell_line_shape": cell_line_consensus_scores.shape,
    "tumor_missing_scores": int(
        tumor_consensus_scores[tumor_score_columns]
        .isna()
        .sum()
        .sum()
    ),
    "cell_line_missing_scores": int(
        cell_line_consensus_scores[cell_line_score_columns]
        .isna()
        .sum()
        .sum()
    ),
    "tumor_score_means": (
        tumor_consensus_scores[tumor_score_columns]
        .mean()
        .round(12)
        .to_dict()
    ),
    "tumor_score_stds": (
        tumor_consensus_scores[tumor_score_columns]
        .std(ddof=1)
        .round(12)
        .to_dict()
    ),
    "cell_line_score_means": (
        cell_line_consensus_scores[cell_line_score_columns]
        .mean()
        .round(12)
        .to_dict()
    ),
    "cell_line_score_stds": (
        cell_line_consensus_scores[cell_line_score_columns]
        .std(ddof=1)
        .round(12)
        .to_dict()
    ),
}

{'tumor_shape': (9965, 8),
 'cell_line_shape': (713, 4),
 'tumor_missing_scores': 0,
 'cell_line_missing_scores': 0,
 'tumor_score_means': {'CONSENSUS_TX_01': -0.0,
  'CONSENSUS_TX_02': 0.0,
  'CONSENSUS_TX_03': 0.0},
 'tumor_score_stds': {'CONSENSUS_TX_01': 1.0,
  'CONSENSUS_TX_02': 1.0,
  'CONSENSUS_TX_03': 1.0},
 'cell_line_score_means': {'CONSENSUS_TX_01': -0.0,
  'CONSENSUS_TX_02': -0.0,
  'CONSENSUS_TX_03': -0.0},
 'cell_line_score_stds': {'CONSENSUS_TX_01': 1.0,
  'CONSENSUS_TX_02': 1.0,
  'CONSENSUS_TX_03': 1.0}}

In [28]:
# =============================================================================
# Align cell-line native and consensus scores for fidelity assessment
# =============================================================================

# Align by ModelID rather than assuming row-order identity between independent
# Phase 3 artifacts.

cell_line_fidelity_scores = (
    cell_line_consensus_scores
    .merge(
        cell_line_native_scores[
            [
                "ModelID",
                *construction_events["cell_line_program"],
            ]
        ],
        on="ModelID",
        how="inner",
        validate="one_to_one",
    )
)

In [29]:
# =============================================================================
# Assess native-score fidelity
# =============================================================================

# Fidelity is descriptive only; no threshold, rescue rule, or reoptimization is
# introduced from these correlations.

native_score_fidelity = pd.DataFrame(
    [
        {
            "consensus_program_id": event.consensus_program_id,
            "tumor_native_score_pearson": pearsonr(
                tumor_consensus_scores[event.consensus_program_id],
                tumor_native_scores[event.tumor_rna_axis],
            ).statistic,
            "tumor_native_score_spearman": spearmanr(
                tumor_consensus_scores[event.consensus_program_id],
                tumor_native_scores[event.tumor_rna_axis],
            ).statistic,
            # Cell-line ICA scores are oriented using the sign frozen in notebook
            # 400; this corrects ICA sign indeterminacy without phenotype guidance.
            "cell_line_native_score_pearson": pearsonr(
                cell_line_fidelity_scores[event.consensus_program_id],
                event.orientation_multiplier
                * cell_line_fidelity_scores[event.cell_line_program],
            ).statistic,
            "cell_line_native_score_spearman": spearmanr(
                cell_line_fidelity_scores[event.consensus_program_id],
                event.orientation_multiplier
                * cell_line_fidelity_scores[event.cell_line_program],
            ).statistic,
        }
        for event in construction_events.itertuples(index=False)
    ]
)

native_score_fidelity

,consensus_program_id,tumor_native_score_pearson,tumor_native_score_spearman,cell_line_native_score_pearson,cell_line_native_score_spearman
0,CONSENSUS_TX_01,0.262281,0.269140,0.653557,0.367358
1,CONSENSUS_TX_02,0.128220,0.120032,0.591080,0.345068
2,CONSENSUS_TX_03,0.236501,0.208687,0.366253,0.222374


In [30]:
# =============================================================================
# Add native-score fidelity to consensus catalog
# =============================================================================

consensus_program_catalog = (
    consensus_program_catalog
    .merge(
        native_score_fidelity,
        on="consensus_program_id",
        how="left",
        validate="one_to_one",
    )
)

In [31]:
# =============================================================================
# Define consensus-program output paths
# =============================================================================

CONSENSUS_CATALOG_PATH = (
    OUTPUT_DIR
    / "401_consensus_transcriptomic_program_catalog.csv"
)

CONSENSUS_GENE_WEIGHTS_PATH = (
    OUTPUT_DIR
    / "401_consensus_transcriptomic_gene_weights.csv"
)

TUMOR_CONSENSUS_SCORES_PATH = (
    OUTPUT_DIR
    / "401_consensus_tumor_scores.parquet"
)

CELL_LINE_CONSENSUS_SCORES_PATH = (
    OUTPUT_DIR
    / "401_consensus_cellline_scores.parquet"
)

TUMOR_ARM_CONTEXT_PATH = (
    OUTPUT_DIR
    / "401_consensus_tumor_arm_context.csv"
)

CONSENSUS_METADATA_PATH = (
    OUTPUT_DIR
    / "401_consensus_construction_metadata.json"
)

In [32]:
# =============================================================================
# Build consensus-construction metadata
# =============================================================================

consensus_construction_metadata = {
    "notebook": "401_consensus_program_construction",
    "representation_scope": (
        "candidate cross-system transcriptomic consensus representation "
        "with tumor-side methylation context"
    ),
    "construction_unit": "unique_supported_rna_correspondence_event",
    "eligible_correspondence_class": "SUPPORTED_CORRESPONDENCE",
    "consensus_program_count": int(len(construction_events)),
    "shared_gene_count": int(len(shared_gene_universe)),
    "gene_universe": (
        "frozen_2389_gene_cross_system_universe_from_notebook_400"
    ),
    "loading_construction": {
        "tumor_loading": "center_then_l2_normalize",
        "cell_line_loading": (
            "orient_by_notebook_400_multiplier_then_center_then_l2_normalize"
        ),
        "system_weighting": "equal_0.5_0.5",
        "final_consensus_loading": "l2_normalized_equal_weight_average",
        "orientation_anchor": "tumor_rna_component",
    },
    "score_construction": {
        "gene_standardization": "within_system_mean0_sd1",
        "projection": "standardized_expression_dot_consensus_loading",
        "final_score_standardization": "within_system_per_program_mean0_sd1",
        "cross_system_joint_normalization": False,
        "absolute_score_scale_cross_system_comparable": False,
    },
    "construction_exclusions": [
        (
            "pharmacogenomic_phenotype_reuse_for_401_"
            "orientation_weighting_scoring_or_rescue"
        ),
        "downstream_biological_annotation",
        "robustness_strength",
        "scientific_priority",
        "tumor_methylation_information",
    ],
    "upstream_candidate_universe_note": (
        "The frozen Phase 4 cell-line candidate space was inherited from Phase 3, "
        "where phenotype association contributed to candidate definition. Notebook "
        "401 does not reuse phenotype for orientation, weighting, score projection, "
        "rescue, or reoptimization."
    ),
    "tumor_methylation_handling": (
        "arm_level_context_only; methylation does not enter transcriptomic "
        "weights or scores and shared RNA events do not fuse methylation arms"
    ),
    "diagnostics": {
        "shared_loading_energy": "descriptive_only_no_threshold_or_gate",
        "native_score_fidelity": "descriptive_only_no_threshold_or_gate",
        "p_values_used": False,
    },
    "ambiguity_policy": (
        "AMBIGUOUS_CORRESPONDENCE events remain authoritative outputs of "
        "notebook 400 and are not constructed or rescued in notebook 401"
    ),
    "inputs": {
        "tumor_handoff": str(
            project_relative_path(TUMOR_HANDOFF_PATH)
        ),
        "correspondence_summary": str(
            project_relative_path(CORRESPONDENCE_SUMMARY_PATH)
        ),
        "shared_gene_universe": str(
            project_relative_path(SHARED_GENE_UNIVERSE_PATH)
        ),
        "tumor_rna_loadings": str(
            project_relative_path(TUMOR_RNA_LOADINGS_PATH)
        ),
        "cell_line_ica_loadings": str(
            project_relative_path(CELL_LINE_ICA_LOADINGS_PATH)
        ),
        "tumor_expression": str(
            project_relative_path(TUMOR_EXPRESSION_PATH)
        ),
        "cell_line_expression": str(
            project_relative_path(CELL_LINE_EXPRESSION_PATH)
        ),
        "cross_system_tumor_arm_handoff": str(
            project_relative_path(CROSS_SYSTEM_TUMOR_ARM_HANDOFF_PATH)
        ),
    },
}

In [33]:
# =============================================================================
# Complete consensus-construction provenance and limitations
# =============================================================================

consensus_construction_metadata["diagnostic_inputs"] = {
    "tumor_native_scores": str(
        project_relative_path(TUMOR_NATIVE_SCORES_PATH)
    ),
    "cell_line_native_scores": str(
        project_relative_path(CELL_LINE_NATIVE_SCORES_PATH)
    ),
}

consensus_construction_metadata["outputs"] = {
    "program_catalog": str(
        project_relative_path(CONSENSUS_CATALOG_PATH)
    ),
    "gene_weights": str(
        project_relative_path(CONSENSUS_GENE_WEIGHTS_PATH)
    ),
    "tumor_scores": str(
        project_relative_path(TUMOR_CONSENSUS_SCORES_PATH)
    ),
    "cell_line_scores": str(
        project_relative_path(CELL_LINE_CONSENSUS_SCORES_PATH)
    ),
    "tumor_arm_context": str(
        project_relative_path(TUMOR_ARM_CONTEXT_PATH)
    ),
    "construction_metadata": str(
        project_relative_path(CONSENSUS_METADATA_PATH)
    ),
}

consensus_construction_metadata["interpretive_limitations"] = [
    (
        "cross-system transcriptomic correspondence does not establish full "
        "epigenetic-transcriptomic reproduction"
    ),
    (
        "tumor arms sharing one RNA component are contextual arms of one "
        "transcriptomic correspondence event, not independent cross-system evidence"
    ),
    (
        "consensus score direction is anchored to the tumor RNA component and "
        "does not encode a resistance-like direction"
    ),
    (
        "system-specific upstream robustness and confounding limitations remain "
        "applicable and are not rehabilitated by consensus construction"
    ),
]

In [34]:
# =============================================================================
# Write consensus-program artifacts
# =============================================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

consensus_program_catalog.to_csv(
    CONSENSUS_CATALOG_PATH,
    index=False,
)

consensus_gene_weights.to_csv(
    CONSENSUS_GENE_WEIGHTS_PATH,
    index=False,
)

tumor_consensus_scores.to_parquet(
    TUMOR_CONSENSUS_SCORES_PATH,
    index=False,
)

cell_line_consensus_scores.to_parquet(
    CELL_LINE_CONSENSUS_SCORES_PATH,
    index=False,
)

tumor_arm_context.to_csv(
    TUMOR_ARM_CONTEXT_PATH,
    index=False,
)

with open(CONSENSUS_METADATA_PATH, "w", encoding="utf-8") as file:
    json.dump(
        consensus_construction_metadata,
        file,
        indent=2,
    )

In [35]:
# =============================================================================
# Verify consensus-program artifact publication
# =============================================================================

consensus_artifact_paths = [
    CONSENSUS_CATALOG_PATH,
    CONSENSUS_GENE_WEIGHTS_PATH,
    TUMOR_CONSENSUS_SCORES_PATH,
    CELL_LINE_CONSENSUS_SCORES_PATH,
    TUMOR_ARM_CONTEXT_PATH,
    CONSENSUS_METADATA_PATH,
]

{
    "artifacts_written": len(consensus_artifact_paths),
    "all_artifacts_exist": all(
        path.exists()
        for path in consensus_artifact_paths
    ),
    "output_directory": str(
        project_relative_path(OUTPUT_DIR)
    ),
}

{'artifacts_written': 6,
 'all_artifacts_exist': True,
 'output_directory': 'data/processed/consensus_programs'}

## Conclusion

Notebook 401 constructed three candidate cross-system transcriptomic consensus
representations from the three independent `SUPPORTED_CORRESPONDENCE` events
frozen by notebook 400:

- `CONSENSUS_TX_01`: `RNA_IC150` ↔ `ICA_PROGRAM_09`
- `CONSENSUS_TX_02`: `RNA_IC151` ↔ `ICA_PROGRAM_29`
- `CONSENSUS_TX_03`: `RNA_IC184` ↔ `ICA_PROGRAM_13`

Each representation retains continuous weights across the full frozen
2,389-gene shared universe. Tumor and cell-line loadings were independently
centered and L2-normalized, combined symmetrically with equal system weighting,
and oriented according to the tumor-anchored sign established in notebook 400.

Consensus scores were generated independently within TCGA and cell lines after
within-system gene standardization and were subsequently standardized
per consensus program within each system. Their absolute magnitudes are
therefore not directly comparable across systems, and positive score direction
does not imply a resistance-like phenotype.

Four tumor cross-omic arms map to the three transcriptomic consensus
representations. In particular, `CROSS_OMIC_PAIR_03` and
`CROSS_OMIC_PAIR_12` remain separate methylation-context arms of the single
`RNA_IC184` transcriptomic correspondence event; they are neither fused nor
counted as independent cross-system transcriptomic evidence.

Shared-loading energy and native-score fidelity were retained as descriptive
diagnostics only. Their heterogeneous values were not used to filter, rank,
reweight, rescue, or reoptimize any consensus representation.

Upstream limitations remain applicable. Consensus construction does not
rehabilitate confounded or context-sensitive source programs and does not
establish full epigenetic-transcriptomic reproduction, causal mechanism,
clinical prediction, or therapeutic relevance.

The resulting artifacts provide the frozen transcriptomic consensus layer for
downstream cross-lineage robustness assessment.